## Energy module - defining a simple trip
In this notebook, we set up graph with a single edge to demonstrate some basic functionality of the Energy module.

We take the following steps:

1. [Imports](#1.-Imports)
2. [Create graph](#2.-Create-graph)
3. [Create vessel](#3.-Create-vessel)
4. [Run simulation](#4.-Run-simulation)
5. [Inspect results](#5.-Inspect-results)

### 1. Imports
We start with importing required libraries

In [2]:
# package(s) used for creating and geo-locating the graph
import networkx as nx  
import shapely.geometry
from shapely.geometry import Point, LineString
from shapely.geometry.base import BaseGeometry
import pyproj


# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime, time
import simpy
import opentnsim
from opentnsim import core as core_module
from opentnsim.energy import mixins as energy_module
from opentnsim import graph as graph_module
from opentnsim import output as output_module
import opentnsim.core.vessel_properties as vessel_module
from opentnsim import vessel_traffic_service as vessel_traffic_service_module

import numpy as np
import pandas as pd

# package(s) needed for plotting
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
# create a basic plotting method (plotly based, with labels properly positioned)
def plot_graph(FG):
    
    def compute_distance(WGS84, origin: shapely.Geometry, destination: shapely.Geometry):
        """Determine the distance based on great circle path from origin to destination."""
        orig = shapely.geometry.shape(origin)
        dest = shapely.geometry.shape(destination)
        _, _, distance = WGS84.inv(orig.x, orig.y, dest.x, dest.y)
    
        return distance
        
    # Extract node positions
    pos = {node: (FG.nodes[node]['geometry'].x, FG.nodes[node]['geometry'].y) for node in FG.nodes}
    
    # Create Plotly figure
    fig = make_subplots(rows=1, cols=1)
    
    # Add edges to the figure
    for edge in FG.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        fig.add_trace(go.Scatter(x=[x0, x1], y=[y0, y1], mode='lines', line=dict(width=4, color='blue')))
    
    
    # Add nodes to the figure
    for node in FG.nodes():
        x, y = pos[node]
        fig.add_trace(go.Scatter(x=[x], y=[y], mode='markers+text', text=['<b> {} </b>'.format(node)], textposition='middle center',
                                 marker=dict(size=20, color='red')))
    
    # Add labels to the figure
    for index, edge in enumerate(FG.edges):
        # Add label '100 km' over the trace with the same y position as the node labels
        mid_x = (pos[edge[0]][0] + pos[edge[1]][0]) / 2
        
        # Use the same y position as the nodes
        mid_y =  pos[edge[0]][1]  

        # Calculate distance
        distance = compute_distance(pyproj.Geod(ellps="WGS84"), FG.nodes[edge[0]]['geometry'], FG.nodes[edge[1]]['geometry'])
        
        # add edge length
        fig.add_trace(go.Scatter(x=[mid_x], y=[mid_y], mode='text', text=['<b>{:.1f} km</b><br>'.format(distance/1000)], 
                                 textposition='top center'))
        # add edge depth
        fig.add_trace(go.Scatter(x=[mid_x], y=[mid_y], mode='text', text=['<br><b> {} </b>'.format(FG.edges[edge]['Info']['GeneralDepth'])],
                                 textposition='bottom center'))
    
    # Update layout
    fig.update_layout(
        {
            'plot_bgcolor':  'rgba(0, 0, 0, 0)',
            'paper_bgcolor': 'rgba(0, 0, 0, 0)',
        },
        margin={"t": 0, "b": 0, "l": 0, "r": 0},
        height=80,
        showlegend=False)
    
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)

    return fig

### 2. Create graph
Next we create a 1D network (a graph) along which the vessel can move. A graph is made of nodes (blue dots in the plot below) and edges (red arrows between the nodes in the plot below). We use the python package networkx to do this. 

For this example, we construct a network of 4 nodes linked by 3 edges. The edges are made bi-directional to allow for two-way traffic, which means that the graph in the end contains 6 edges.

In [4]:
# Create a one edge graph (distance 100 km)
node_A = graph_module.Node(name='0', geometry=Point(0, 0))
node_B = graph_module.Node(name='1', geometry=Point(1 * 0.8983152841195216, 0))

nodes  = [node_A, node_B]
edges  = [(node_A, node_B)]
depths = {"GeneralDepth": [10.0]}

FG = graph_module.DiGraph(edges=edges, edges_info=depths).graph

In [5]:
plot_graph(FG)

### 3. Create vessel
Initiate the simpy environment. Create a vessel class. We call this class a *Vessel*, and add a number of OpenTNSim mix-ins to this class. Each mix-in requires certain input parameters. 

The following mix-ins are sufficient to create a vessel for our problem: 
* _ConsumesEnergy_ - enables calculation of resistance, required power and emissions (from the energy_module)
* _IsVessel_ - allows to give the vessel specific properties (from the vessel_module) 

In [6]:
# Start simpy environment
t_start = datetime.datetime(2024, 1, 1, 0, 0, 0)

env = simpy.Environment(initial_time=t_start.timestamp())

env.epoch = t_start
env.simulation_start = t_start

In [7]:
# Add the graph to environment
env.graph = FG

# In order from ships to know where they are going we need to add an VTS to the environment
#env.vessel_traffic_service = vessel_traffic_service_module.VesselTrafficService(env=env)

In [8]:

class IsVessel(core_module.Identifiable,
               core_module.Movable,
               vessel_module.VesselProperties,
               core_module.ExtraMetadata,
               graph_module.HasMultiDiGraph,
               output_module.HasOutput):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        
Vessel = type('Vessel',(energy_module.ConsumesEnergy,   # The vessel consumes energy
                        IsVessel,         # Basic information of the vessel
                       ),{})  

In [9]:
# instantiate vessel object with the following inputs
route = nx.dijkstra_path(FG, source='0', target='1')
geometry = FG.nodes['0']['geometry']
vessel = Vessel(**{ "env": env,
                    "name": 'Vessel',           # you can give the vessel an arbitratry name
                    "origin": '0',              # start node of the route to sail
                    "destination": '1',         # stop node of the route to sail
                    "type": 'Va/M9 - Verl. Groot Rijnschip', # This indicates the vessel class. This info is mainly informative.
                    "L": 135,                   # m
                    "B": 11.45,                 # m
                    "T": 2.75,                  # m
                    "v": 5,                     # m/s If None: this value is calculated based on P_tot_given
                    "safety_margin": 0.2,       # for tanker vessel with sandy bed the safety margin is recommended as 0.2 m 
                    "h_squat": False,           # if the ship should squat while moving, set to True, otherwise set to False
                    "P_installed": 1750.0,      # kW
                    "P_tot_given": None,        # kW If None: this value is calculated value based on speed
                    "bulbous_bow": False,       # if a vessel has no bulbous_bow, set to False; otherwise set to True.
                    "P_hotel_perc": 0.05,       # 0: all power goes to propulsion
                    "P_hotel": None,            # None: calculate P_hotel from percentage
                    "x": 2,                     # number of propellers
                    "L_w": 3.0 ,
                    "C_B": 0.85,                # block coefficient 
                    "C_year": 1990,             # engine build year
                    "arrival_time": datetime.datetime(2024, 1, 1, 0, 0, 0),
                    "geometry": geometry,
                    "route": route,             # the route to sail
                  }
               )

env.process(vessel.move())

<Process(move) object at 0x1cb5b0aecf0>

### 4. Run simulation
Now we can run the simulation.

In [10]:
env.run()

### 5. Inspect results

In [ ]:
# the logging information is found in the logbook
pd.DataFrame.from_dict(vessel.logbook)

,Message,Timestamp,Value,Geometry
0,Sailing from node 0 to node 1 start,2024-01-01 00:00:00,"{'origin': '0', 'destination': '1', 'route': [...",POINT (0 0)
1,Sailing from node 0 to node 1 stop,2024-01-01 05:33:20,"{'origin': '0', 'destination': '1', 'route': [...",POINT (0.8983152841195216 0)


In [ ]:
# execute the calculation of energy consumption and emissions by post-processing the vessel log info
energycalculation = energy_module.EnergyCalculation(env.graph, vessel)
energycalculation.calculate_energy_consumption()

In [ ]:
# process the data frame to get the right output values
df = pd.DataFrame.from_dict(energycalculation.energy_use)
df['fuel_kg_per_km'] = (df['total_diesel_consumption_ICE_mass'] / 1000) / (df['distance']/1000)
df['CO2_g_per_km']   = (df['total_emission_CO2']) / (df['distance']/1000)
df['PM10_g_per_km']  = (df['total_emission_PM10']) / (df['distance']/1000)
df['NOx_g_per_km']   = (df['total_emission_NOX']) / (df['distance']/1000)
df

,time_start,time_stop,edge_start,edge_stop,P_tot,P_given,P_installed,total_energy,total_diesel_consumption_C_year_ICE_mass,total_diesel_consumption_ICE_mass,...,total_emission_PM10,total_emission_NOX,stationary,water depth,distance,delta_t,fuel_kg_per_km,CO2_g_per_km,PM10_g_per_km,NOx_g_per_km
0,2024-01-01,2024-01-01 05:33:20,POINT (0 0),POINT (0.8983152841195216 0),1162.327962,1162.327962,1750.0,6457.377568,1.434829e+06,1.470697e+06,...,2557.121517,64382.259146,6457.377568,10.0,100000.0,20000.0,14.706966,45523.220379,25.571215,643.822591


In [ ]:
print('The time difference is: {} seconds'.format((df.time_stop.item() - df.time_start.item()).seconds))

The time difference is: 20000 seconds
